In [ ]:
import os
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

from protossl.datasets import (
    HeedbECGDataset,
    EchoNextECGDataset,
    MimicECGDataset,
    ZzuECGDataset,
    PtbxlECGDataset,
    get_ptbxl_labels,
    CincECGDataset,
    Code15ECGDataset,
)
from protossl.defines import (
    HEEDB_TARGETS, 
    ECHONEXT_TARGETS,
    MIMIC_TARGETS,
    ZZU_TARGETS,
    PTBXL_TARGETS,
    CINC_TARGETS,
    CODE15_TARGETS,
)


import pandas as pd
import numpy as np
from typing import Optional


class TableOne:
    """
    A simplified, fast version of TableOne for large datasets.

    Parameters
    ----------
    data : pd.DataFrame
        Input data.
    columns : list
        Columns to include in the table.
    categorical : list
        Categorical columns.
    continuous : list
        Continuous columns.
    nonnormal : list, optional
        Continuous columns to display as median [Q1, Q3] instead of mean (SD).
    nunique : list, optional
        Columns for which to report the number of unique values per group
        (e.g. patient ID, study ID). Rendered as a single row per column.
    groupby : str, optional
        Column to group by (becomes column headers).
    order : dict, optional
        Mapping of column name -> ordered list of values. Used for groupby column
        ordering and for ordering categorical row values.
    binary_show : dict, optional
        Mapping of binary categorical column name -> the single class to display.
        Only columns explicitly listed here will be collapsed to a single row;
        binary columns not listed are rendered with all levels like any other
        categorical.
    """

    def __init__(
        self,
        data: pd.DataFrame,
        columns: list,
        categorical: Optional[list] = None,
        continuous: Optional[list] = None,
        nonnormal: Optional[list] = None,
        nunique: Optional[list] = None,
        groupby: Optional[str] = None,
        order: Optional[dict] = None,
        binary_show: Optional[dict] = None,
    ):
        self.data = data
        self.columns = list(columns)
        self.categorical = list(categorical) if categorical else []
        self.continuous = list(continuous) if continuous else []
        self.nonnormal = set(nonnormal) if nonnormal else set()
        self.nunique = list(nunique) if nunique else []
        self.groupby = groupby
        self.order = order or {}
        self.binary_show = binary_show or {}

        # Don't render the groupby column as a row — it's a diagonal submatrix.
        if self.groupby is not None:
            self.categorical = [c for c in self.categorical if c != self.groupby]
            self.continuous = [c for c in self.continuous if c != self.groupby]
            self.nunique = [c for c in self.nunique if c != self.groupby]
            self.columns = [c for c in self.columns if c != self.groupby]

        # Make sure nunique columns appear in self.columns even if the user
        # didn't list them explicitly.
        for c in self.nunique:
            if c not in self.columns:
                self.columns.append(c)

        self.table = self._build()

    # ------------------------------------------------------------------ #
    # Group setup
    # ------------------------------------------------------------------ #
    def _group_indexers(self):
        """Return list of (group_label, boolean_mask, n) tuples."""
        if self.groupby is None:
            mask = np.ones(len(self.data), dtype=bool)
            return [("Overall", mask, int(mask.sum()))]

        col = self.data[self.groupby]
        if self.groupby in self.order:
            levels = list(self.order[self.groupby])
        else:
            levels = list(pd.unique(col.dropna()))

        out = []
        for lvl in levels:
            mask = (col == lvl).to_numpy()
            out.append((lvl, mask, int(mask.sum())))
        return out

    # ------------------------------------------------------------------ #
    # Continuous formatting
    # ------------------------------------------------------------------ #
    @staticmethod
    def _fmt_mean_sd(x: np.ndarray) -> str:
        x = x[~np.isnan(x)]
        if x.size == 0:
            return ""
        return f"{x.mean():.1f} ({x.std(ddof=1):.1f})"

    @staticmethod
    def _fmt_median_iqr(x: np.ndarray) -> str:
        x = x[~np.isnan(x)]
        if x.size == 0:
            return ""
        q1, med, q3 = np.quantile(x, [0.25, 0.5, 0.75])
        return f"{med:.1f} [{q1:.1f}, {q3:.1f}]"

    def _continuous_row(self, col: str, groups):
        nonnormal = col in self.nonnormal
        label = f"{col}, {'median [Q1, Q3]' if nonnormal else 'mean (SD)'}"
        values = self.data[col].to_numpy(dtype=float, na_value=np.nan)

        row = {}
        for gname, mask, _ in groups:
            sub = values[mask]
            row[gname] = self._fmt_median_iqr(sub) if nonnormal else self._fmt_mean_sd(sub)
        return label, row

    # ------------------------------------------------------------------ #
    # Unique-count formatting
    # ------------------------------------------------------------------ #
    def _nunique_row(self, col: str, groups):
        label = f"{col}, unique n"
        series = self.data[col].to_numpy()
        row = {}
        for gname, mask, _ in groups:
            sub = series[mask]
            # pd.unique handles NaN, mixed types, and is faster than np.unique
            # on object dtype.
            row[gname] = f"{pd.unique(sub).size}"
        return label, row

    # ------------------------------------------------------------------ #
    # Categorical formatting
    # ------------------------------------------------------------------ #
    def _categorical_rows(self, col: str, groups):
        """
        Yields (label, {group: 'n (pct%)'} ) tuples.
        Only collapses to a single row if the user has explicitly listed the
        column in binary_show.
        """
        series = self.data[col]

        # Explicit single-class selection — only path that collapses the column.
        if col in self.binary_show:
            chosen = self.binary_show[col]
            label = f"{col} = {chosen}, n (%)"
            row = {}
            chosen_mask = (series == chosen).to_numpy()
            for gname, mask, n in groups:
                if n == 0:
                    row[gname] = ""
                    continue
                count = int((chosen_mask & mask).sum())
                pct = 100.0 * count / n
                row[gname] = f"{count} ({pct:.1f})"
            yield label, row
            return

        # Determine level order
        if col in self.order:
            levels = list(self.order[col])
        else:
            levels = sorted(series.dropna().unique().tolist(), key=lambda v: (str(type(v)), v))

        # Header row + one row per level
        yield f"{col}, n (%)", {gname: "" for gname, _, _ in groups}
        for lvl in levels:
            label = f"    {lvl}"
            row = {}
            lvl_mask = (series == lvl).to_numpy()
            for gname, mask, n in groups:
                if n == 0:
                    row[gname] = ""
                    continue
                count = int((lvl_mask & mask).sum())
                pct = 100.0 * count / n
                row[gname] = f"{count} ({pct:.1f})"
            yield label, row

    # ------------------------------------------------------------------ #
    # Build
    # ------------------------------------------------------------------ #
    def _build(self) -> pd.DataFrame:
        groups = self._group_indexers()
        group_names = [g[0] for g in groups]

        rows = []
        index = []

        # n row
        index.append("n")
        rows.append({gname: f"{n}" for gname, _, n in groups})

        nunique_set = set(self.nunique)

        # Preserve user-specified column order
        for col in self.columns:
            if col in nunique_set:
                label, row = self._nunique_row(col, groups)
                index.append(label)
                rows.append(row)
            elif col in self.continuous:
                label, row = self._continuous_row(col, groups)
                index.append(label)
                rows.append(row)
            elif col in self.categorical:
                for label, row in self._categorical_rows(col, groups):
                    index.append(label)
                    rows.append(row)

        df = pd.DataFrame(rows, index=index, columns=group_names)
        df.columns = pd.MultiIndex.from_tuples(
            [(self.groupby or "", c) for c in df.columns]
        )
        return df

    # ------------------------------------------------------------------ #
    # Display
    # ------------------------------------------------------------------ #
    def __repr__(self) -> str:
        return self.table.to_string()

    def _repr_html_(self) -> str:
        return self.table.to_html()

    def to_csv(self, path, **kwargs):
        return self.table.to_csv(path, **kwargs)

    def to_latex(self, **kwargs):
        return self.table.to_latex(**kwargs)

In [ ]:
ecg_dir = Path("/opt/gpudata/ecg")

datasets = {
    "EchoNext": {
        72475: "echonext",
        32768: "echonext-32k",
        16384: "echonext-16k",
        8192: "echonext-8k",
        4096: "echonext-4k",
        2048: "echonext-2k",
        1024: "echonext-1k",
        512: "echonext-512",
        256: "echonext-256",
    },
    "MIMIC-IV-ECG": {
        78470: "mimic-iv-ecg",
        32768: "mimic-iv-ecg-32k",
        16384: "mimic-iv-ecg-16k",
        8192: "mimic-iv-ecg-8k",
        4096: "mimic-iv-ecg-4k",
        2048: "mimic-iv-ecg-2k",
        1024: "mimic-iv-ecg-1k",
        512: "mimic-iv-ecg-512",
        256: "mimic-iv-ecg-256",
    },
    "CODE-15%": {
        74112: "code15",
        32768: "code15-32k",
        16384: "code15-16k",
        8192: "code15-8k",
        4096: "code15-4k",
        2048: "code15-2k",
        1024: "code15-1k",
        512: "code15-512",
        256: "code15-256",
    },
    "PTB-XL": {
        17418: "ptb-xl",
        8722: "ptb-xl-8k",
        4356: "ptb-xl-4k",
        2175: "ptb-xl-2k",
        1091: "ptb-xl-1k",
        547: "ptb-xl-512",
        273: "ptb-xl-256",
    },
    "CinC Georgia": {
        8192: "cinc-2020",
        4096: "cinc-2020-4k",
        2048: "cinc-2020-2k",
        1024: "cinc-2020-1k",
        512: "cinc-2020-512",
        256: "cinc-2020-256",
    },
    "ZZU pECG": {
        8658: "zzu-pecg",
        4096: "zzu-pecg-4k",
        2048: "zzu-pecg-2k",
        1024: "zzu-pecg-1k",
        512: "zzu-pecg-512",
        256: "zzu-pecg-256",
    },
}

### HEEDB

In [ ]:
if not os.path.exists("temp_heedb_train.csv"):
    ds_train = HeedbECGDataset(
        dataset_path=str(ecg_dir / "heedb"),
        split="train",
        sampling_rate=100,
    )
    ds_val = HeedbECGDataset(
        dataset_path=str(ecg_dir / "heedb"),
        split="val",
        sampling_rate=100,
    )
    ds_test = HeedbECGDataset(
        dataset_path=str(ecg_dir / "heedb"),
        split="test",
        sampling_rate=100,
    )

    train_df = ds_train._df.copy()
    val_df = ds_val._df.copy()
    test_df = ds_test._df.copy()

    train_df[list(HEEDB_TARGETS)] = ds_train.labels
    val_df[list(HEEDB_TARGETS)] = ds_val.labels
    test_df[list(HEEDB_TARGETS)] = ds_test.labels

    train_df[["patient_id", "age", "sex", "source", "split"] + list(HEEDB_TARGETS)].to_csv("temp_heedb_train.csv", index=False)
    val_df[["patient_id", "age", "sex", "source", "split"] + list(HEEDB_TARGETS)].to_csv("temp_heedb_val.csv", index=False)
    test_df[["patient_id", "age", "sex", "source", "split"] + list(HEEDB_TARGETS)].to_csv("temp_heedb_test.csv", index=False)

train_df = pd.read_csv("temp_heedb_train.csv")
val_df = pd.read_csv("temp_heedb_val.csv")
test_df = pd.read_csv("temp_heedb_test.csv")

df = pd.concat([train_df, val_df, test_df], ignore_index=True)
df["sex"] = df["sex"].str.lower()
df.columns = ["Patient"] + list(df.columns.str.title())[1:]

tab1 = TableOne(
    data=df,
    columns=list(df.columns),
    nunique=["Patient"],
    categorical=[c for c in df.columns if c != "Age" and c != "Patient"],
    continuous=["Age"],
    nonnormal=["Age"],
    groupby="Split",
    order={
        "Split": list(df["Split"].drop_duplicates()),
    },
    binary_show={
        k.title(): 1 for k in HEEDB_TARGETS
    },
)

print(tab1.to_latex())

### EchoNext

In [ ]:
ds_key = "EchoNext"
make_dataset = lambda ds_dir, split: EchoNextECGDataset(dataset_path=str(ecg_dir/ds_dir), split=split, sampling_rate=100)
def get_df(ds_dir, split):
    ds = make_dataset(ds_dir, split)
    df = ds._df[["patient_key", "age_at_ecg", "sex", "split"] + list(ECHONEXT_TARGETS)].copy()
    df.columns = ["Patient", "Age", "Sex", "Split"] + list(ECHONEXT_TARGETS)
    if split == "train":
        suffix = ds_dir.split("echonext")[-1]
        if suffix == "":
            suffix = "Full"
        else:
            suffix = suffix[1:] # remove leading dash
        df["Split"] = f"Train ({suffix})"
    else:
        df["Split"] = split.title()
    return df


dfs_train = []
df_val = None
df_test = None
for i, (train_size, ds_dir) in enumerate(datasets[ds_key].items()):
    df_train = get_df(ds_dir, "train")
    assert len(df_train) == train_size
    dfs_train.append(df_train)
    if i == 0:
        df_val = get_df(ds_dir, "val")
        df_test = get_df(ds_dir, "test")

df = pd.concat(dfs_train + [df_val, df_test], ignore_index=True)

tab1 = TableOne(
    data=df,
    columns=list(df.columns),
    nunique=["Patient"],
    categorical=[c for c in df.columns if c != "Age" and c != "Patient"],
    continuous=["Age"],
    nonnormal=["Age"],
    groupby="Split",
    order={
        "Split": list(df["Split"].drop_duplicates()),
    },
    binary_show={
        k: 1 for k in ECHONEXT_TARGETS
    },
)

print(tab1.to_latex())

### MIMIC

In [ ]:
ds_key = "MIMIC-IV-ECG"
make_dataset = lambda ds_dir, split: MimicECGDataset(dataset_path=str(ecg_dir/ds_dir), split=split, sampling_rate=100)
def get_df(ds_dir, split):
    ds = make_dataset(ds_dir, split)
    df = ds._df[["subject_id", "age", "gender", "split"] + MIMIC_TARGETS].copy()
    df.columns = ["Patient", "Age", "Sex", "Split"] + MIMIC_TARGETS
    if split == "train":
        suffix = ds_dir.split("mimic-iv-ecg")[-1]
        if suffix == "":
            suffix = "Full"
        else:
            suffix = suffix[1:] # remove leading dash
        df["Split"] = f"Train ({suffix})"
    else:
        df["Split"] = split.title()
    return df


dfs_train = []
df_val = None
df_test = None
for i, (train_size, ds_dir) in enumerate(datasets[ds_key].items()):
    df_train = get_df(ds_dir, "train")
    assert len(df_train) == train_size
    dfs_train.append(df_train)
    if i == 0:
        df_val = get_df(ds_dir, "val")
        df_test = get_df(ds_dir, "test")

df = pd.concat(dfs_train + [df_val, df_test], ignore_index=True)

tab1 = TableOne(
    data=df,
    columns=list(df.columns),
    nunique=["Patient"],
    categorical=[c for c in df.columns if c != "Age" and c != "Patient"],
    continuous=["Age"],
    nonnormal=["Age"],
    groupby="Split",
    order={
        "Split": list(df["Split"].drop_duplicates()),
    },
    binary_show={
        k: 1 for k in MIMIC_TARGETS
    },
)

print(tab1.to_latex())

### ZZU

In [ ]:
ds_key = "ZZU pECG"
make_dataset = lambda ds_dir, split: ZzuECGDataset(dataset_path=str(ecg_dir/ds_dir), split=split, sampling_rate=100)
def get_df(ds_dir, split):
    ds = make_dataset(ds_dir, split)
    df = ds._df[["Patient_ID", "Age", "Gender", "split"] + list(ZZU_TARGETS)].copy()
    df.columns = ["Patient", "Age", "Sex", "Split"] + list(ZZU_TARGETS)
    df["Age"] = df["Age"].str.strip("d").astype(int) # age in days for ZZU
    if split == "train":
        suffix = ds_dir.split("zzu-pecg")[-1]
        if suffix == "":
            suffix = "Full"
        else:
            suffix = suffix[1:] # remove leading dash
        df["Split"] = f"Train ({suffix})"
    else:
        df["Split"] = split.title()
    return df


dfs_train = []
df_val = None
df_test = None
for i, (train_size, ds_dir) in enumerate(datasets[ds_key].items()):
    df_train = get_df(ds_dir, "train")
    assert len(df_train) == train_size
    dfs_train.append(df_train)
    if i == 0:
        df_val = get_df(ds_dir, "val")
        df_test = get_df(ds_dir, "test")

df = pd.concat(dfs_train + [df_val, df_test], ignore_index=True)

tab1 = TableOne(
    data=df,
    columns=list(df.columns),
    nunique=["Patient"],
    categorical=[c for c in df.columns if c != "Age" and c != "Patient"],
    continuous=["Age"],
    nonnormal=["Age"],
    groupby="Split",
    order={
        "Split": list(df["Split"].drop_duplicates()),
    },
    binary_show={
        k: 1 for k in ZZU_TARGETS
    },
)

print(tab1.to_latex())

### PTB-XL

In [ ]:
ds_key = "PTB-XL"
make_dataset = lambda ds_dir, split: PtbxlECGDataset(dataset_path=str(ecg_dir/ds_dir), split=split, sampling_rate=100)
def get_df(ds_dir, split):
    ds = make_dataset(ds_dir, split)
    ds._df[PTBXL_TARGETS] = get_ptbxl_labels(ds._df)
    ds._df["split"] = split
    df = ds._df[["patient_id", "age", "sex", "split"] + PTBXL_TARGETS].copy()
    df.columns = ["Patient", "Age", "Sex", "Split"] + PTBXL_TARGETS
    if split == "train":
        suffix = ds_dir.split("ptb-xl")[-1]
        if suffix == "":
            suffix = "Full"
        else:
            suffix = suffix[1:] # remove leading dash
        df["Split"] = f"Train ({suffix})"
    else:
        df["Split"] = split.title()
    return df


dfs_train = []
df_val = None
df_test = None
for i, (train_size, ds_dir) in enumerate(datasets[ds_key].items()):
    df_train = get_df(ds_dir, "train")
    assert len(df_train) == train_size
    dfs_train.append(df_train)
    if i == 0:
        df_val = get_df(ds_dir, "val")
        df_test = get_df(ds_dir, "test")

df = pd.concat(dfs_train + [df_val, df_test], ignore_index=True)

tab1 = TableOne(
    data=df,
    columns=list(df.columns),
    nunique=["Patient"],
    categorical=[c for c in df.columns if c != "Age" and c != "Patient"],
    continuous=["Age"],
    nonnormal=["Age"],
    groupby="Split",
    order={
        "Split": list(df["Split"].drop_duplicates()),
    },
    binary_show={
        k: 1 for k in PTBXL_TARGETS
    },
)

print(tab1.to_latex())

### CinC

In [ ]:
ds_key = "CinC Georgia"
make_dataset = lambda ds_dir, split: CincECGDataset(dataset_path=str(ecg_dir/ds_dir), split=split, sampling_rate=100)
def get_df(ds_dir, split):
    ds = make_dataset(ds_dir, split)
    df = ds._df[["patient_id", "age", "sex", "split"] + CINC_TARGETS].copy()
    df.columns = ["Patient", "Age", "Sex", "Split"] + CINC_TARGETS
    if split == "train":
        suffix = ds_dir.split("cinc-2020")[-1]
        if suffix == "":
            suffix = "Full"
        else:
            suffix = suffix[1:] # remove leading dash
        df["Split"] = f"Train ({suffix})"
    else:
        df["Split"] = split.title()
    return df


dfs_train = []
df_val = None
df_test = None
for i, (train_size, ds_dir) in enumerate(datasets[ds_key].items()):
    df_train = get_df(ds_dir, "train")
    assert len(df_train) == train_size
    dfs_train.append(df_train)
    if i == 0:
        df_val = get_df(ds_dir, "val")
        df_test = get_df(ds_dir, "test")

df = pd.concat(dfs_train + [df_val, df_test], ignore_index=True)

tab1 = TableOne(
    data=df,
    columns=list(df.columns),
    nunique=["Patient"],
    categorical=[c for c in df.columns if c != "Age" and c != "Patient"],
    continuous=["Age"],
    nonnormal=["Age"],
    groupby="Split",
    order={
        "Split": list(df["Split"].drop_duplicates()),
    },
    binary_show={
        k: 1 for k in CINC_TARGETS
    },
)

print(tab1.to_latex())

### CODE-15%

In [ ]:
ds_key = "CODE-15%"
make_dataset = lambda ds_dir, split: Code15ECGDataset(dataset_path=str(ecg_dir/ds_dir), split=split, sampling_rate=100)
def get_df(ds_dir, split):
    ds = make_dataset(ds_dir, split)
    df = ds._df[["patient_id", "age", "is_male", "split"] + CODE15_TARGETS].copy()
    df.columns = ["Patient", "Age", "Sex", "Split"] + CODE15_TARGETS
    if split == "train":
        suffix = ds_dir.split("code15")[-1]
        if suffix == "":
            suffix = "Full"
        else:
            suffix = suffix[1:] # remove leading dash
        df["Split"] = f"Train ({suffix})"
    else:
        df["Split"] = split.title()
    return df


dfs_train = []
df_val = None
df_test = None
for i, (train_size, ds_dir) in enumerate(datasets[ds_key].items()):
    df_train = get_df(ds_dir, "train")
    assert len(df_train) == train_size
    dfs_train.append(df_train)
    if i == 0:
        df_val = get_df(ds_dir, "val")
        df_test = get_df(ds_dir, "test")

df = pd.concat(dfs_train + [df_val, df_test], ignore_index=True)

tab1 = TableOne(
    data=df,
    columns=list(df.columns),
    nunique=["Patient"],
    categorical=[c for c in df.columns if c != "Age" and c != "Patient"],
    continuous=["Age"],
    nonnormal=["Age"],
    groupby="Split",
    order={
        "Split": list(df["Split"].drop_duplicates()),
    },
    binary_show={
        k: 1 for k in CODE15_TARGETS
    },
)

print(tab1.to_latex())